# Topic embeddings in 2D with t-SNE

TO-DO:
- Try different random seeds
- Try differetn perplexity values

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.manifold import TSNE
from sklearn.preprocessing import normalize

DATA_DIR = Path("my_data/sgns")
PLOT_DIR = Path("plots")
RANDOM_STATE = 42
PERPLEXITY = 30 # effective number of neighbors
MAX_ITER = 1_000
TOPIC_COLUMNS = ["Topic_1", "Topic_2", "Topic_3"]

## 1. Load the topic vectors

In [2]:
topics_df = pd.read_csv(
    DATA_DIR / "topic_embeddings_normalized.csv", keep_default_na=False,
    dtype={"Topic_Id": "string", **{column: "string" for column in TOPIC_COLUMNS}},
)
embedding_columns = sorted(column for column in topics_df if column.startswith("embedding_"))
normalized_embeddings = topics_df[embedding_columns].to_numpy(dtype=np.float32)

print(f"Loaded {len(topics_df):,} topics with {normalized_embeddings.shape[1]} dimensions")

Loaded 1,000 topics with 150 dimensions


## 2. Run t-SNE

In [3]:
tsne = TSNE(
    n_components=2,
    perplexity=PERPLEXITY,
    learning_rate="auto",
    init="pca",
    metric="euclidean",
    max_iter=MAX_ITER,
    random_state=RANDOM_STATE,
)
coordinates = tsne.fit_transform(normalized_embeddings)
projection_df = topics_df[["Topic_Id", *TOPIC_COLUMNS]].copy()
projection_df["Topic_path"] = projection_df[TOPIC_COLUMNS].apply(
    lambda row: " / ".join(value for value in row if value), axis=1
)
projection_df[["TSNE_1", "TSNE_2"]] = coordinates
print(f"Final KL divergence: {tsne.kl_divergence_:.4f}")

Final KL divergence: 1.2226


## 3. Plot and save

Output:

- `plots/topic_embeddings_tsne.html`: interactive plot that can be opened in a browser.

In [ ]:
fig = px.scatter(
    projection_df,
    x="TSNE_1", y="TSNE_2", color="Topic_1",
    hover_name="Topic_path",
    hover_data={"Topic_Id": False, "Topic_1": False, "TSNE_1": False, "TSNE_2": False},
    title=f"Topic embeddings · t-SNE (perplexity={PERPLEXITY}, seed={RANDOM_STATE})",
    labels={"Topic_1": "Main topic", "TSNE_1": "t-SNE 1", "TSNE_2": "t-SNE 2"},
    template="plotly_white",
    height=800,
)
fig.update_traces(marker=dict(size=7, opacity=0.8))
fig.update_layout(legend_title_text="Main topic")
PLOT_DIR.mkdir(parents=True, exist_ok=True)
# projection_df.to_csv(DATA_DIR / "topic_embeddings_tsne.csv", index=False)
fig.write_html(PLOT_DIR / "topic_embeddings_tsne.html", include_plotlyjs=True)
fig.show()